In [ ]:
# --- Dataset ---
class HandSegmentationDataset(Dataset):
    def __init__(self, json_path, img_dir, transform=None):
        import json
        with open(json_path) as f:
            self.tasks = json.load(f)
        self.img_dir = img_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.tasks)
        
    def __getitem__(self, idx):
        task = self.tasks[idx]
        img_name = Path(task['image']).name # Adjust based on Label Studio format
        img_path = self.img_dir / img_name
        image = np.array(Image.open(img_path).convert('RGB'))
        h, w = image.shape[:2]
        
        # Generate mask from polygons (Simplified)
        mask = np.zeros((h, w), dtype=np.uint8)
        
        # NOTE: You need to parse Label Studio polygon format here
        # This is pseudo-code for parsing:
        if 'polygon' in task:
            for poly in task['polygon']:
                 points = np.array(poly['points']) # [[x,y], [x,y]...] relative 0-100
                 points[:, 0] = points[:, 0] * w / 100
                 points[:, 1] = points[:, 1] * h / 100
                 cv2.fillPoly(mask, [points.astype(int)], 1)
                 
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
        image = transforms.ToTensor()(image)
        mask = torch.from_numpy(mask).unsqueeze(0).float() # (1, H, W)
        return image, mask

# --- Model ---
model = smp.Unet(
    encoder_name="efficientnet-b0", 
    classes=1, 
    activation='sigmoid'
).to(device)

# --- Training ---
# Dice Loss is often better for thin segmentation like hands
criterion = smp.losses.DiceLoss(mode='binary')
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop similar to Keypoints...